<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/WHSAT_text_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets scikit-learn torch

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/main/cleaned_safety_data_set_B.csv")

# Keep only necessary columns
data_set = df[['description', 'severity', 'potential_severity', 'critical_risk', 'near_miss_gap']].dropna()
data_set.head()

,description,severity,potential_severity,critical_risk,near_miss_gap
0,While removing the drill rod of the Jumbo 08 f...,1,4,Pressed,3
1,During the activation of a sodium sulphide pum...,1,4,Pressurized Systems,3
2,In the sub-station MILPO located at level +170...,1,3,Manual Tools,2
3,Being 9:45 am. approximately in the Nv. 1880 C...,1,1,Others,0
4,Approximately at 11:45 a.m. in circumstances t...,4,4,Others,0


In [ ]:
#creating a dictionary with text label into numerical value. Eg: "Bees"==1
data_set['critical_risk_label'] = data_set['critical_risk'].astype('category').cat.codes
risk_labels = dict(enumerate(data_set['critical_risk'].astype('category').cat.categories))
print(risk_labels)

{0: '\nNot applicable', 1: 'Bees', 2: 'Blocking and isolation of energies', 3: 'Burn', 4: 'Chemical substances', 5: 'Confined space', 6: 'Cut', 7: 'Electrical Shock', 8: 'Electrical installation', 9: 'Fall', 10: 'Fall prevention', 11: 'Fall prevention (same level)', 12: 'Individual protection equipment', 13: 'Liquid Metal', 14: 'Machine Protection', 15: 'Manual Tools', 16: 'Others', 17: 'Plates', 18: 'Poll', 19: 'Power lock', 20: 'Pressed', 21: 'Pressurized Systems', 22: 'Pressurized Systems / Chemical Substances', 23: 'Projection', 24: 'Projection of fragments', 25: 'Projection/Burning', 26: 'Projection/Choco', 27: 'Projection/Manual Tools', 28: 'Suspended Loads', 29: 'Traffic', 30: 'Vehicles and Mobile Equipment', 31: 'Venomous Animals', 32: 'remains of choco'}


In [ ]:
#to make 0–4
#needed for classification models which expect 0-indexed labels
data_set['severity_label'] = data_set['severity'] - 1
data_set['severity_label'].value_counts()

,count
severity_label,
0,316
1,40
2,31
3,30
4,8


In [ ]:
#splitting the data set as trainning category (80%) and testing category (20%)
#ensures severity distribution is balanced in both sets (important for imbalanced data)
train_df, test_df = train_test_split(data_set, test_size=0.2, stratify=data_set['severity_label'], random_state=42)

In [ ]:
#tokenizing the description into IDs
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=False,
    )


Pay more attention from here onwards

In [39]:
from torch.utils.data import Dataset
#Converts everything to PyTorch tensors
class SafetyDataset(Dataset):
    def __init__(self, texts, labels, severity, potential_severity, near_miss_gap):
        self.encodings = tokenize(texts) #This converts all your text descriptions into numbers that the model can understand
        self.labels = list(labels)
        self.severity = list(severity)
        self.potential_severity = list(potential_severity)
        self.near_miss_gap = list(near_miss_gap)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        item['severity'] = torch.tensor(self.severity[idx])
        item['potential_severity'] = torch.tensor(self.potential_severity[idx])
        item['near_miss_gap'] = torch.tensor(self.near_miss_gap[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [41]:
train_dataset = SafetyDataset(train_df['description'], train_df['critical_risk_label'], train_df['severity'], train_df['potential_severity'], train_df['near_miss_gap'])
test_dataset = SafetyDataset(test_df['description'], test_df['critical_risk_label'], test_df['severity'], test_df['potential_severity'], test_df['near_miss_gap'])

num_labels = data_set['critical_risk_label'].nunique()
print(num_labels)

33


In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,          # keep small for CPU
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_dir='./logs',
    save_strategy="no"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1": f1_score(labels, preds, average='macro')
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

save_dir = "/content/drive/MyDrive/WHSAT/"
os.makedirs(save_dir, exist_ok=True)

# Get predictions from the trained model
predictions = trainer.predict(test_dataset)
preds = predictions.predictions
y_test = predictions.label_ids

np.save(os.path.join(save_dir, "transformer_probs.npy"), preds)
np.save(os.path.join(save_dir, "y_test.npy"), y_test)
np.save(os.path.join(save_dir, "test_indices.npy"), test_df.index.values)

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
